# One-time final frozen test evaluation

**Scientific question:** What are the final held-out metrics and paired test-set bootstrap intervals for the four frozen models?
**Configuration:** the immutable four-model registry created by Notebook 09.
**Dataset:** locked 1,927-image clean dataset.
**Split:** the sole 231-sample held-out test, opened only after full preflight.
**Checkpoint:** four validation-selected checkpoints with registry-locked SHA256 values.
**Expected outputs:** per-sample predictions, final metrics/tables, paired bootstrap files, paper figures, and `FINAL_TEST_LOCK.json` under `results/final_test/`.

This notebook performs no training, tuning, checkpoint selection, optimizer creation, backward pass, or `model.train()` call.


## 1. Imports and machine-local data root

Resolve the configured raw-data root without creating any dataset or loader.


In [1]:
from pathlib import Path
import sys

REPO = Path.cwd().resolve()
while REPO != REPO.parent and not (REPO / "pyproject.toml").is_file():
    REPO = REPO.parent
if not (REPO / "pyproject.toml").is_file():
    raise RuntimeError("Open this notebook from inside the SoilNet repository")
sys.path.insert(0, str(REPO / "src"))

import json
from soilnet.final_sequence import preflight_frozen_registry, run_final_test_once
from soilnet.io import resolve_paths


## 2. Complete frozen-registry preflight

Before test construction, verify exactly four immutable models, exact paths/hashes, split identity/count metadata, validation artifacts, and absence of a completed test lock.


In [2]:
registry = preflight_frozen_registry()
assert registry["model_count"] == 4 and registry["TEST_OPENED"] == "NO"
print(json.dumps({"preflight": "PASS", "models": [m["experiment_id"] for m in registry["models"]], "test_loader": "NOT_CREATED_YET"}, indent=2))


{
  "preflight": "PASS",
  "models": [
    "P0_FINAL_SOILNET_VICREG_MU27_LI_V4_BESTREG",
    "P1_SOILNET_VICREG_MU27_NO_LI_BESTREG",
    "P1_SOILNET_IMAGENET_LI_NO_SSL_BESTREG",
    "P2_MOBILEVITV2_IMAGENET_LI_BESTREG"
  ],
  "test_loader": "NOT_CREATED_YET"
}


## 3. Evaluate once and lock

Only after the preceding preflight passes, create one shared test dataset/loader, evaluate all four frozen models under inference mode, persist predictions, bootstrap by paired sample index, plot from saved prediction CSVs, and write the final lock last.


In [3]:
paths = resolve_paths()
final_lock = run_final_test_once(paths["data_root"])
assert final_lock["TEST_EVALUATED"] == "YES" and final_lock["test_n"] == 231
print(json.dumps(final_lock, indent=2))


{
  "TEST_OPENED": "YES",
  "TEST_EVALUATED": "YES",
  "MODEL_SET_FROZEN": "YES",
  "timestamp": "2026-09-06T15:14:43.937637+00:00",
  "split_sha256": "8927b8223b8c4c234d264ad9ea62ac2df6161124a79787a2e71eeb5cd23eae2f",
  "test_n": 231,
  "model_sha256": {
    "P0": "eba009dfd45ec21174a8e40b16148e0455e902286c7db55933487221da761379",
    "P1_noLI": "a9d8de995b0673e9ec39bfc4afac00d6a2a773ad9ffbea0027ff2d0c4fab820c",
    "P1_noSSL": "50c0f15567ab71a87b04569790b9cc7afd016989e3899d56e1898d8a6fcfb044",
    "P2_MobileViTv2": "0a080d5129a2e57f1c2baf420b5af9aadf913fc0aab0d0050b6d662f24c28a8d"
  },
  "software_environment": {
    "platform": "Linux-6.18.33.2-microsoft-standard-WSL2-x86_64-with-glibc2.39",
    "python": "3.11.15",
    "torch": "2.6.0+cu118",
    "torchvision": "0.21.0+cu118",
    "timm": "1.0.29",
    "numpy": "2.4.6",
    "cuda_available": true,
    "cuda_runtime": "11.8",
    "device": "NVIDIA GeForce RTX 3050"
  },
  "bootstrap": {
    "seed": 20260906,
    "n_bootstrap": 10000